# Pelatihan Sistem Deteksi Anemia di Google Colab

Perbandingan **Pipeline 1 (YOLOv8n + U-Net)**, **Pipeline 2 (MediaPipe Face Mesh)**, dan
**Pipeline 2b (MediaPipe MagicTouch)** dengan classifier MobileNetV3-Small, memakai dataset Eyes-defy-anemia.

**Langkah:**
1. Unggah `archive.zip` (dataset Eyes-defy-anemia) ke Google Drive: `MyDrive/anemia-detection-colab/archive.zip`.
2. Menu **Runtime → Change runtime type → T4 GPU → Save**.
3. Menu **Runtime → Run all**, lalu izinkan akses Google Drive saat diminta.
4. Semua hasil (model, laporan, grafik, log) tersimpan otomatis di
   `MyDrive/anemia-detection-colab/hasil/<RUN_NAME>/` setiap kali satu tahap selesai.

**Bila sesi terputus:** buka lagi notebook ini, ubah `START_STEP` ke tahap yang belum selesai
(lihat baris terakhir di `log_pipeline.txt`), biarkan `RUN_NAME` sama, lalu **Run all** lagi.
Hasil tahap yang sudah selesai diambil kembali dari Drive.

**Bila Colab meminta "Restart session" setelah instalasi paket:** klik restart, lalu **Run all** sekali lagi.

In [ ]:
# ===== Pengaturan =====
RUN_NAME   = "run_utama"   # nama folder hasil di Drive; samakan untuk melanjutkan run yang terputus
START_STEP = "1"           # 1, 2, 3a, 3b, 4, 5, 6, 7
BRANCH     = "dev"
DRIVE_DIR  = "/content/drive/MyDrive/anemia-detection-colab"

In [ ]:
# ===== Cek GPU =====
import torch
assert torch.cuda.is_available(), "GPU belum aktif. Pilih Runtime > Change runtime type > T4 GPU, lalu Run all lagi."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ===== Hubungkan Google Drive =====
import os
from google.colab import drive
drive.mount("/content/drive")
ZIP = f"{DRIVE_DIR}/archive.zip"
OUT = f"{DRIVE_DIR}/hasil/{RUN_NAME}"
assert os.path.exists(ZIP), f"archive.zip belum ada. Unggah dataset Eyes-defy-anemia ke {DRIVE_DIR}/ terlebih dahulu."
os.makedirs(OUT, exist_ok=True)
print("Hasil akan disimpan di:", OUT)

In [ ]:
# ===== Ambil kode dari GitHub =====
REPO = "/content/anemia-detection"
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/Hatif4706/anemia-detection.git {REPO}
else:
    !git -C {REPO} pull -q origin {BRANCH}
%cd {REPO}
!git log --oneline -1

In [ ]:
# ===== Instal library (PyTorch + CUDA sudah tersedia di Colab) =====
!pip install -q ultralytics==8.4.174 segmentation-models-pytorch==0.5.0 mediapipe==1.1.0 openpyxl xlrd timm

In [ ]:
# ===== Siapkan dataset dan model MediaPipe =====
import pathlib, shutil, zipfile, urllib.request
dst = pathlib.Path("data/raw/eyes_defy_anemia")
if not dst.exists():
    tmp = pathlib.Path("/content/tmp_eda")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(tmp)
    src = next(p for p in tmp.rglob("Italy") if p.is_dir()).parent  # folder yang berisi Italy/ dan India/
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(src), str(dst))
    shutil.rmtree(tmp, ignore_errors=True)
print("Dataset:", sorted(p.name for p in dst.iterdir()))

pathlib.Path("models").mkdir(exist_ok=True)
MP_MODELS = {
    "face_landmarker.task": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
    "magic_touch.tflite": "https://storage.googleapis.com/mediapipe-models/interactive_segmenter/magic_touch/float32/1/magic_touch.tflite",
}
for name, url in MP_MODELS.items():
    f = pathlib.Path("models") / name
    if not f.exists():
        urllib.request.urlretrieve(url, f)
    print(name, f.stat().st_size // 1024, "KB")

In [ ]:
# ===== Jalankan pipeline per tahap; hasil disalin ke Drive setelah setiap tahap =====
import subprocess, sys

STEPS = ["1", "2", "3a", "3b", "4", "5", "6", "7"]
LOG = f"{OUT}/log_pipeline.txt"

def sync_to_drive():
    for d in ["models", "reports", "runs"]:
        if os.path.exists(d):
            subprocess.run(["rsync", "-a", d, OUT + "/"], check=False)

def run_step(s):
    with open(LOG, "a", encoding="utf-8") as log:
        log.write(f"\n##### TAHAP {s} #####\n")
        p = subprocess.Popen([sys.executable, "-u", "scripts/run_pipeline.py", "--start", s, "--stop", s],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            if "iCCP" in line:   # peringatan profil warna PNG dari dataset, aman diabaikan
                continue
            print(line, end="")
            log.write(line)
        p.wait()
    sync_to_drive()
    if p.returncode != 0:
        raise RuntimeError(f"Tahap {s} gagal. Lihat log di atas, lalu set START_STEP = '{s}' dan Run all lagi.")
    print(f"--> Tahap {s} selesai dan tersimpan di Drive")

# Ambil kembali hasil tahap sebelumnya dari Drive (untuk melanjutkan run yang terputus)
for d in ["models", "reports", "runs"]:
    if os.path.exists(f"{OUT}/{d}"):
        subprocess.run(["rsync", "-a", f"{OUT}/{d}", "./"], check=False)

start = STEPS.index(START_STEP)
# Data olahan tidak disimpan di Drive (cepat dibuat ulang), jadi siapkan lagi bila sesi baru
if start > 0 and not os.path.exists("data/splits/test.csv"):
    run_step("1")
if START_STEP == "3a" and not os.path.exists("data/processed/yolo/data.yaml"):
    run_step("2")
if START_STEP == "5" and not os.path.exists("data/processed/rois"):
    start = STEPS.index("4")

for s in STEPS[start:]:
    run_step(s)
print("\nSELESAI. Semua hasil ada di Google Drive:", OUT)

In [ ]:
# ===== Ringkasan hasil =====
from IPython.display import Image, Markdown, display
display(Markdown(open("reports/results_table.md", encoding="utf-8").read()))
for f in ["anotasi_bbox_dari_mask.png", "kurva_pelatihan_unet.png", "contoh_roi_kedua_pipeline.png"]:
    p = f"reports/figures/{f}"
    if os.path.exists(p):
        print(f)
        display(Image(p, width=900))
print("Laporan progres (buka dari Drive):", f"{OUT}/reports/progress/laporan_progres.html")